# 직접 준비한 640×640 ROI → 심볼 라벨 → YOLO 학습 → CAD BLOCK

**PDF에서 640×640 영역을 고르는 작업은 사용자가 직접 합니다.** 1-1번 셀에서 로컬 자르기 화면을 열고 도면을 확대해 직접 위치를 선택·저장하세요. 이미 잘라 둔 이미지가 있다면 바로 `prepared_rois` 폴더에 넣어도 됩니다. 자동으로 심볼 위치를 고르거나 임의로 사진을 저장하지 않습니다.

1. 직접 자른 PNG/JPG를 `prepared_rois` 폴더에 넣습니다.
2. 이미지 크기가 정확히 640×640인지 검사합니다.
3. 각 이미지 안의 대상 심볼에 박스/클래스를 표시하거나 이미 만든 YOLO 라벨을 가져옵니다.
4. YOLO11s detect를 `imgsz=640`으로 학습합니다.
5. 학습 후 원본 PDF에서 심볼을 검출하고 검토합니다.
6. CAD에서 PDFIMPORT 후 검토한 영역을 BLOCK으로 묶는 스크립트를 실행합니다.

**학습 이미지 640×640과 정답 박스는 별개입니다.** 한 ROI에 심볼이 여러 개라면 모두 표시하세요. 아직 심볼 학습을 실행하거나 전체 학습 데이터 이미지를 만들어 둔 상태가 아닙니다. OpenAI API는 사용하지 않습니다.

## 0. 처음 한 번 설치

VS Code에서 Python 3.11 가상환경을 커널로 선택하세요. 아래 설치 명령의 `#`을 지워 한 번 실행하고 커널을 재시작하세요. 이미지 폴더만 있어서는 검출 학습이 되지 않으며 3번 또는 4번에서 정답 박스가 필요합니다.

In [ ]:
# 처음 한 번만 실행 후 커널 재시작
# %pip install -r requirements.txt
# 상위 프로젝트 폴더에서 실행한다면:
# %pip install -r symbol_roi640/requirements.txt
import sys
print("현재 커널:", sys.executable)

## 1. 폴더·클래스 설정

`PREPARED_ROI_DIR`에 직접 자른 640×640 사진을 넣으세요. 기본 위치는 이 노트북 옆 `prepared_rois`입니다. 형태/핀 수가 다른 심볼은 다른 클래스로 정하세요. `CLASSES` 순서는 라벨 파일의 클래스 번호와 같아야 합니다.

원본 PDF 경로는 학습 입력이 아니라 **학습 후 전체 심볼 검출**에 사용합니다. `DPI` 역시 추후 PDF 검출 해상도입니다. 직접 만든 crop의 렌더링 DPI를 모르면 아래 값이 맞다고 가정할 수 없습니다. crop마다 같은 PDF 확대율을 쓰고, 검출 시 심볼 픽셀 크기가 학습 이미지와 비슷하도록 DPI를 조정하세요.

In [ ]:
from pathlib import Path
import os, sys, json, math
from IPython.display import display, clear_output

HERE=Path.cwd()
PROJECT=HERE if (HERE/"roi_tools.py").is_file() else HERE/"symbol_roi640"
if not (PROJECT/"roi_tools.py").is_file():
    raise FileNotFoundError("노트북과 roi_tools.py가 들어 있는 symbol_roi640 폴더에서 실행하세요.")
sys.path.insert(0,str(PROJECT.resolve()))
from roi_tools import *

PREPARED_ROI_DIR=PROJECT/"prepared_rois"  # 직접 자른 사진 폴더
PREPARED_LABEL_DIR=PROJECT/"prepared_labels"  # 이미 라벨링했다면 txt 파일 폴더
CLASSES=["symbol_A","symbol_B"]  # 실제 심볼 형태별 이름으로 수정
DATA_ROOT=PROJECT/"my_dataset_prepared"
OUTPUT=PROJECT/"output"
PRETRAINED=PROJECT/"weights"/"yolo11s.pt"
DEVICE="cpu"
PDF_PATH=Path(r"C:\Users\Win11Pro\Desktop\1. DWG 객체 변환 도면 생성 샘플\DWG 객체 변환 샘플.pdf")
PAGE=1
DPI=150  # 학습 후 원본 PDF 검출 시 조정

ROI_GROUPS={
    # 같은 심볼 위치를 중복해서 자른 사진들은 같은 그룹으로 지정하세요.
    # "roi_001.png":"same_location_A", "roi_002.png":"same_location_A",
}
for folder in (PREPARED_ROI_DIR,PREPARED_LABEL_DIR,OUTPUT): folder.mkdir(exist_ok=True)
os.environ["YOLO_CONFIG_DIR"]=str(PROJECT/".yolo_config")
os.environ["YOLO_AUTOINSTALL"]="false"
os.environ["YOLO_OFFLINE"]="true"
if PDF_PATH.is_file():
    info=pdf_info(PDF_PATH)[PAGE-1]
    page_size_pt=[info["width_pt"],info["height_pt"]]
else:
    page_size_pt=None
print("직접 자른 이미지 넣을 곳:",PREPARED_ROI_DIR.resolve())

## 1-1. PDF를 보면서 직접 640×640 자르기

아래 셀을 실행하고 링크를 열면 PDF를 직접 확대·이동하면서 ROI 위치를 고를 수 있습니다. 「640 ROI 선택」 상태에서 심볼 중심을 클릭하거나 박스를 드래그하세요. 오른쪽 실제 640×640 사진을 확인한 후 **저장 버튼**을 누릅니다. 저장 전에는 학습 이미지 파일이 생기지 않습니다.

파일은 `prepared_rois`에 PNG로 저장되고, 좌표/DPI는 같은 이름의 JSON에 기록됩니다. 이 화면은 OCR/YOLO 검출을 실행하지 않습니다. 원하는 사진을 저장한 뒤 노트북으로 돌아와 2번 셀을 다시 실행하세요. 해상도와 프레임이 맞지 않으면 저장 DPI를 바꾸고 다시 선택하세요.

In [ ]:
from crop_launcher import start_crop_ui
from IPython.display import HTML
CROP_URL=start_crop_ui(PDF_PATH,PREPARED_ROI_DIR,port=8510)
display(HTML(f'<a href="{CROP_URL}" target="_blank">PDF에서 직접 640×640 영역 자르기 화면 열기</a>'))
print("자른 사진 저장 폴더:",PREPARED_ROI_DIR.resolve())

## 2. 직접 만든 640×640 사진 확인

정확히 640×640인 파일만 통과합니다. 크기가 다르면 파일명을 표시하고 중단합니다. 원본 사진은 변경하지 않습니다. 파일이 없으면 먼저 위 폴더에 사진을 넣고 이 셀을 다시 실행하세요.

In [ ]:
ROI_FILES=prepared_roi_files(PREPARED_ROI_DIR)
print("사용자가 준비한 ROI:",len(ROI_FILES),"장")
for image_path in ROI_FILES[:12]:
    print(image_path.name)
    image,_=load_prepared_roi(image_path)
    display(image)
if not ROI_FILES:
    print("아직 ROI 사진이 없습니다. prepared_rois 폴더에 직접 자른 사진을 넣으세요.")

## 3. 사진 안의 심볼 라벨링

**이 화면에서 PDF를 자르는 것이 아닙니다.** 드롭다운으로 이미 준비한 사진을 선택하고, 클래스 선택 후 사진 안의 심볼 박스를 드래그합니다. 여러 심볼을 모두 표시하고 검토 완료 후 저장하세요. 다음 사진으로 이동하기 전에 저장하세요. 심볼이 없는 음성 샘플도 검토 완료 후 빈 라벨로 저장할 수 있습니다.

`%matplotlib widget` 오류가 나면 0번에서 `ipympl` 설치 후 커널을 재시작하세요. 이미 LabelImg 등에서 YOLO 라벨을 만들었다면 이 셀을 건너뛰고 4번을 사용하세요.

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
from matplotlib.widgets import RectangleSelector
from matplotlib.patches import Rectangle
import ipywidgets as widgets

if not ROI_FILES:
    raise ValueError("2번에서 확인할 640×640 이미지가 없습니다.")
plt.close("all")
plt.rcParams["font.family"]="Malgun Gothic"
plt.rcParams["axes.unicode_minus"]=False
fig,ax_roi=plt.subplots(figsize=(8,8))
image_pick=widgets.Dropdown(options=[(p.name,str(p)) for p in ROI_FILES],description="사진")
class_pick=widgets.Dropdown(options=[(name,i) for i,name in enumerate(CLASSES)],description="클래스")
reviewed=widgets.Checkbox(value=False,description="이 사진의 대상 심볼 전체 라벨 검토 완료")
undo_button=widgets.Button(description="마지막 박스 취소")
save_button=widgets.Button(description="라벨 저장",button_style="success")
message=widgets.Output()
state={"image":None,"meta":None,"boxes":[],"artists":[]}

def redraw():
    for artist in state["artists"]: artist.remove()
    state["artists"]=[]
    if ax_roi.images: ax_roi.images[0].set_data(state["image"])
    else: ax_roi.imshow(state["image"])
    for box in state["boxes"]:
        x1,y1,x2,y2=box["xyxy"]
        rectangle=Rectangle((x1,y1),x2-x1,y2-y1,fill=False,edgecolor="red",linewidth=2)
        ax_roi.add_patch(rectangle)
        label=ax_roi.text(x1,y1,CLASSES[box["class_id"]],color="red")
        state["artists"].extend([rectangle,label])
    ax_roi.set_xlim(0,640);ax_roi.set_ylim(640,0)
    ax_roi.set_title(f"{Path(image_pick.value).name} / 640×640 / 박스 {len(state['boxes'])}")
    fig.canvas.draw_idle()

def load_image(_=None):
    path=Path(image_pick.value)
    image,meta=load_prepared_roi(path,group=ROI_GROUPS.get(path.name))
    boxes=[]
    manifest_path=DATA_ROOT/"annotations.json"
    if manifest_path.is_file():
        manifest=json.loads(manifest_path.read_text(encoding="utf-8"))
        if manifest["classes"]!=CLASSES: raise ValueError("저장된 클래스 순서와 CLASSES가 다릅니다.")
        for roi in manifest["rois"]:
            if roi.get("source_image")==meta["source_image"]: boxes=roi["boxes"]
    state.update(image=image,meta=meta,boxes=boxes)
    reviewed.value=False
    redraw()

def select_box(start,end):
    if start.xdata is None or end.xdata is None: return
    x1,x2=sorted([max(0,min(640,start.xdata)),max(0,min(640,end.xdata))])
    y1,y2=sorted([max(0,min(640,start.ydata)),max(0,min(640,end.ydata))])
    if x2-x1>=2 and y2-y1>=2:
        state["boxes"].append({"class_id":class_pick.value,"xyxy":[x1,y1,x2,y2]})
        reviewed.value=False
        redraw()

def undo(_):
    if state["boxes"]: state["boxes"].pop()
    reviewed.value=False
    redraw()

def save(_):
    with message:
        clear_output(wait=True)
        try:
            item=save_roi(DATA_ROOT,state["image"],state["meta"],state["boxes"],CLASSES,reviewed=reviewed.value)
            print("저장:",Path(image_pick.value).name,"심볼",len(item["boxes"]),"개")
        except Exception as exc: print("저장 안 됨:",exc)

load_image()
selector=RectangleSelector(ax_roi,select_box,useblit=False,button=[1],minspanx=2,minspany=2,spancoords="pixels")
image_pick.observe(load_image,names="value")
undo_button.on_click(undo);save_button.on_click(save)
display(widgets.HBox([image_pick,class_pick]),widgets.HBox([undo_button,save_button]),reviewed,message)
plt.show()

## 4. 선택 사항: 이미 만든 YOLO 라벨 가져오기

`prepared_rois/roi_001.png`와 `prepared_labels/roi_001.txt`처럼 이름을 맞추세요. txt는 `class_id cx cy width height`(0~1) 형식입니다. 대상 심볼이 없는 사진은 빈 txt가 필요합니다. 빠진 라벨을 음성 샘플로 임의 처리하지 않습니다. 3번에서 라벨링했다면 아래는 `False`로 둡니다.

In [ ]:
IMPORT_EXISTING_LABELS=False
if IMPORT_EXISTING_LABELS:
    count=import_prepared_labels(PREPARED_ROI_DIR,PREPARED_LABEL_DIR,DATA_ROOT,CLASSES,groups=ROI_GROUPS)
    print("사용자 이미지/라벨 가져오기:",count,"쌍")

## 5. 데이터셋 분리 및 점검

준비한 이미지와 라벨을 train/val로 분리합니다. 같은 이미지의 복제와 `ROI_GROUPS`에 지정한 같은 위치 그룹은 함께 배치합니다. **외부에서 자른 사진의 PDF 원점은 알 수 없어 겹치는 ROI를 자동 판단할 수 없습니다.** 같은 위치를 중복해서 자른 파일은 1번에서 같은 그룹으로 지정하세요.

정답 박스 없는 파일은 먼저 라벨링하세요. 같은 PDF의 val 점수는 다른 PDF 성능이 아닙니다. 클래스가 한쪽에 빠지면 해당 클래스의 다른 위치 샘플을 더 추가하세요.

In [ ]:
DATA_YAML, split_stats = export_dataset(DATA_ROOT, val_fraction=.2, seed=42)
print("데이터셋:", DATA_YAML)
print(json.dumps(split_stats,ensure_ascii=False,indent=2))
print(DATA_YAML.read_text(encoding="utf-8"))

## 6. 사전학습 YOLO11s 준비 (선택 다운로드)

**YOLO11s detect**를 기본으로 선택했습니다. 형태별 심볼 위치 검출용이며 선을 다시 벡터화하는 모델이 아닙니다. 회사 이미지가 아닌 공개 사전학습 가중치만 다운로드하는 선택 셀입니다. 이미 받은 `yolo11s.pt`를 `weights` 폴더에 놓아도 됩니다. 다운로드 셀 실행 후에는 학습/검출을 오프라인으로 돌립니다.

[Ultralytics detect 데이터 형식](https://docs.ultralytics.com/datasets/detect/) · [검출/학습 공식 문서](https://docs.ultralytics.com/tasks/detect/)

In [ ]:
DOWNLOAD_PRETRAINED = False  # 필요하면 True로 바꾸고 이 셀만 실행
if DOWNLOAD_PRETRAINED:
    from urllib.request import urlretrieve
    PRETRAINED.parent.mkdir(exist_ok=True)
    if not PRETRAINED.exists():
        urlretrieve("https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11s.pt",PRETRAINED)
    print("받은 가중치:",PRETRAINED)
else:
    print("사전학습 가중치 존재:",PRETRAINED.is_file(),PRETRAINED)

## 7. 학습

ROI 라벨을 충분히 수집한 뒤 `RUN_TRAIN=True`로 바꾸세요. 우선 클래스당 여러 실제 위치·방향의 심볼을 수집하고, 배선만 있는 음성 ROI도 넣으세요. 한 페이지에서 심볼 개수가 부족하면 다른 도면이 필요합니다.

라벨 없는 crop, 라벨 잘린 crop, 같은 crop 복제는 학습 품질을 떨어뜨립니다. 회전/대칭이 BLOCK 방향과 연결되면 flip 증강은 끄세요. 아래는 `workers=0`, CPU, `amp=False`로 Windows에서 먼저 실행을 확인하는 설정입니다.

In [ ]:
RUN_TRAIN = False
EPOCHS = 100
BATCH = 4
if RUN_TRAIN:
    from ultralytics import YOLO, settings
    from unittest.mock import patch
    settings.update({key:False for key in ["sync","hub","wandb","mlflow","clearml","comet","neptune","dvc","tensorboard"]
                     if key in settings})
    if not PRETRAINED.is_file(): raise FileNotFoundError("6번에서 로컬 사전학습 가중치를 준비하세요.")
    if "DATA_YAML" not in globals(): raise ValueError("5번 데이터셋 분리부터 실행하세요.")
    def block_network(*args,**kwargs):
        raise PermissionError("학습 중 네트워크 호출을 차단했습니다.")
    # 회사 도면 학습 중 패키지/모델 자동 다운로드와 외부 통신 차단.
    with patch("socket.socket.connect",block_network), patch("socket.create_connection",block_network):
        model=YOLO(str(PRETRAINED),task="detect")
        training=model.train(
            data=str(DATA_YAML),imgsz=640,epochs=EPOCHS,batch=BATCH,
            device=DEVICE,workers=0,amp=False,cache=False,
            project=str(PROJECT/"runs"),name="symbols640",exist_ok=False,
            seed=42,patience=25,degrees=0,fliplr=0,flipud=0,
            mosaic=.3,close_mosaic=10,translate=.05,scale=.15,
            hsv_h=0,hsv_s=0,hsv_v=.05,
        )
    BEST_WEIGHTS=Path(model.trainer.best)
    print("학습 결과:",BEST_WEIGHTS)
else:
    print("ROI 라벨을 준비한 뒤 RUN_TRAIN=True로 바꿔 실행하세요.")

## 8. 검증 점수와 오류 이미지

학습 이후 val의 precision/recall/mAP와 confusion matrix를 확인하세요. 필요한 클래스가 val에 없으면 그 클래스 검증은 불가능합니다. mAP가 높아도 같은 PDF 기준일 수 있으므로 별도 PDF를 최종 평가에 사용하세요.

In [ ]:
RUN_VAL = False
BEST_WEIGHTS = globals().get("BEST_WEIGHTS", PROJECT/"runs"/"symbols640"/"weights"/"best.pt")
if RUN_VAL:
    from ultralytics import YOLO, settings
    settings.update({key:False for key in ["sync","hub","wandb","mlflow","clearml","comet","neptune","dvc","tensorboard"]
                     if key in settings})
    validator=YOLO(str(BEST_WEIGHTS),task="detect")
    metrics=validator.val(data=str(DATA_YAML),imgsz=640,device=DEVICE,workers=0,plots=True)
    print(metrics.results_dict)
    print("검증 이미지:",metrics.save_dir)
else:
    print("학습 후 RUN_VAL=True로 변경하세요.")

## 9. 전체 도면을 640 타일로 검출

PDF 전체를 작은 한 장으로 줄이지 않습니다. **150 DPI의 640 타일**을 160픽셀씩 겹쳐 검출하고, 동일 클래스의 중복 박스를 전체 좌표에서 합칩니다. 타일 경계에 걸린 심볼은 잘릴 수 있어 `tile_edge=True`로 별도 검토합니다.

처음에는 `REGION_PT=[왼쪽,위,오른쪽,아래]`로 배선 그림 영역만 지정하면 부품표 오검출과 시간을 줄일 수 있습니다. 아래 기본값은 `None`(페이지 전체)입니다. 이 PDF의 전면 탐색은 타일 수가 많습니다. 박스 안의 선·픽셀을 YOLO가 객체별로 분해하는 것은 아니며 실제 BLOCK 형상은 별도로 준비합니다.

In [ ]:
RUN_INFERENCE = False
REGION_PT = None  # 예: [0, 0, 5000, 2300]. 미리보기 좌표를 보고 직접 설정
CONFIDENCE = .25
if RUN_INFERENCE:
    from unittest.mock import patch
    from ultralytics import settings
    settings.update({key:False for key in ["sync","hub","wandb","mlflow","clearml","comet","neptune","dvc","tensorboard"]
                     if key in settings})
    progress=widgets.IntProgress(value=0,min=0,max=1,description="타일 검출")
    display(progress)
    def progress_cb(done,total):
        progress.max=total;progress.value=done
    def deny_network(*args,**kwargs):
        raise PermissionError("회사 PDF 검출 중 네트워크 호출을 차단했습니다.")
    with patch("socket.socket.connect",deny_network),patch("socket.create_connection",deny_network):
        DETECTION_REPORT=infer_pdf(
            PDF_PATH,BEST_WEIGHTS,CLASSES,page=PAGE,dpi=DPI,
            region_pt=REGION_PT,overlap=160,conf=CONFIDENCE,device=DEVICE,on_progress=progress_cb)
    write_json(OUTPUT/"detections.json",DETECTION_REPORT)
    print("검출:",len(DETECTION_REPORT["detections"]),"타일:",DETECTION_REPORT["tile_count"])
else:
    print("학습 완료 후 RUN_INFERENCE=True로 실행하세요.")

## 10. 심볼별 확대 검토 → 승인

검출된 심볼을 원본 crop과 대조합니다. 이름이 같아도 다른 형태가 섞여 있으면 클래스나 라벨을 고쳐 재학습하세요. `approved=False`가 기본이며, 확인한 ID만 승인하세요. `rotation_deg`는 CAD BLOCK의 회전각(반시계, 도)이며 detect 모델이 추정한 값이 아닙니다.

In [ ]:
if (OUTPUT/"detections.json").is_file():
    DETECTION_REPORT=json.loads((OUTPUT/"detections.json").read_text(encoding="utf-8"))
    for detection,crop in detection_crops(PDF_PATH,DETECTION_REPORT):
        print(detection["id"],detection["class_name"],round(detection["confidence"],3),
              "타일 경계:",detection["tile_edge"],"PDF bbox:",detection["bbox_pt_top"])
        display(crop)
else:
    print("아직 학습된 모델의 실제 검출 결과가 없습니다.")

In [ ]:
APPROVED_IDS = []  # 예: ["SYMBOL_0001", "SYMBOL_0003"]. 확인한 것만 추가
ROTATIONS = {}      # 예: {"SYMBOL_0003": 90.0}. CAD BLOCK 기준 회전각
if "DETECTION_REPORT" in globals():
    for detection in DETECTION_REPORT["detections"]:
        detection["approved"]=detection["id"] in APPROVED_IDS
        detection["rotation_deg"]=ROTATIONS.get(detection["id"],0.0)
    write_json(OUTPUT/"detections.reviewed.json",DETECTION_REPORT)
    print("승인:",len(APPROVED_IDS))

## 11. CAD BLOCK 생성 / 라이브러리 교체 스크립트

**기본 `group_imported` 모드:** YOLO 검출 영역 안의 PDFIMPORT 객체들을 복사해 새 BLOCK 정의를 만들고, 그 BLOCK을 같은 위치에 삽입한 후 원본 선택 객체를 제거합니다. 클래스/심볼 ID가 포함된 BLOCK 이름이 생성됩니다. 심볼의 실제 선·색상·모양을 활용하므로 미리 준비한 BLOCK 라이브러리가 필요 없습니다. 객체마다 BLOCK 정의가 하나씩 생깁니다. 같은 클래스의 서로 다른 도형을 강제로 같은 모양으로 바꾸지 않습니다.

**선택 `replace_library` 모드:** 검출 위치를 회사의 표준 BLOCK으로 교체합니다. 이 모드는 아래 둘 중 하나로 BLOCK을 미리 준비해야 합니다.

- CAD에서 기준 심볼의 실제 선/곡선을 선택하고 `BLOCK`으로 만듭니다.
- 회사의 기존 심볼 BLOCK 라이브러리를 DWG에 삽입해 정의를 불러옵니다.

라이브러리 모드의 `reference_width`는 **BLOCK 정의의 회전 전 가로 폭(CAD 단위)**이고, `anchor`는 검출 박스에서 BLOCK 기준점이 놓일 비율입니다. `[0.5,0.5]`는 중앙 기준점인 BLOCK만 해당합니다. 핀/케이블 접속점을 기준점으로 쓸 경우 그에 맞게 수정하세요. 회전은 검토한 0/90/180/270도만 지원합니다. 기본 묶기 모드에서는 원본 방향/축척을 유지합니다.

PDFIMPORT로 가져온 배선이 한 폴리라인으로 심볼 안팎을 이어가는 경우 그 객체 전체를 지우면 배선이 사라집니다. 생성된 LISP는 **박스 안에 완전히 포함된 객체만**, 지정한 import layer에서 선택하고, 화면으로 확인한 뒤 심볼별 `Yes`를 입력해야 교체합니다. 박스 안에 완전히 들어간 짧은 배선은 여전히 선택될 수 있습니다. 교체 후보를 확인하고 필요하면 `BREAK`로 경계를 나누거나 `Skip`하세요.

캘리브레이션은 원본 PDF의 두 점과 CAD에서 대응되는 두 점을 사용합니다. 아래 기본은 페이지의 좌하단/우상단입니다. 페이지 모서리가 CAD에서 잡히지 않으면 **그림 테두리의 같은 두 교점**을 PDF 미리보기 좌표로 지정하세요. 두 점은 X/Y 모두 떨어져 있어야 합니다. PDFIMPORT는 회전 0°, X/Y 같은 축척으로 가져옵니다.

In [ ]:
BLOCK_MODE = "group_imported"  # 기본: PDFIMPORT 객체를 그대로 BLOCK으로 묶기
# BLOCK_MODE = "replace_library"  # 선택: 표준 BLOCK 라이브러리로 교체
IMPORT_LAYER = "PDF_Geometry"  # CAD 실제 레이어 이름 확인 (ASCII 이름)
BLOCK_MAP = {
    # 이름/폭/기준점을 실제 준비한 BLOCK에 맞춰 작성하세요.
    # "symbol_A": {"block_name":"CONNECTOR_A","reference_width":10.0,
    #              "anchor":[.5,.5],"import_layer":"PDF_Geometry"},
}
SOURCE_POINTS_PT = [
    [0.0, page_size_pt[1]],  # A: PDF 페이지 좌하단
    [page_size_pt[0], 0.0],  # B: PDF 페이지 우상단
]
MAKE_LISP = False
if MAKE_LISP:
    reviewed_report=json.loads((OUTPUT/"detections.reviewed.json").read_text(encoding="utf-8"))
    lisp_path=make_block_lisp(reviewed_report,BLOCK_MAP,OUTPUT/"replace_symbols.lsp",
                              source_points=SOURCE_POINTS_PT,
                              mode=BLOCK_MODE,import_layer=IMPORT_LAYER)
    print("스크립트:",lisp_path)
    print("CAD에서 APPLOAD → SYMBOL640 → 기준점 A/B 선택 → 심볼별 교체 승인 → SAVEAS")
else:
    print("검출 승인/레이어 확인 후 MAKE_LISP=True로 실행하세요. 라이브러리 모드는 BLOCK_MAP도 필요합니다.")

## 12. CAD에서 PDFIMPORT → BLOCK 교체 → 새 DWG 저장

1. AutoCAD/ZWCAD에서 새 도면을 엽니다. 원본 DWG가 있다면 원본 객체를 활용하는 편이 더 정확합니다.
2. `PDFIMPORT`로 이 PDF의 1페이지를 가져옵니다. 벡터 geometry를 가져오고 회전 0°, 균일 축척을 사용하세요. **Import as block은 끄세요.** 켠 경우 가져온 전체 BLOCK을 확인 후 `EXPLODE`해야 개별 선을 선택할 수 있습니다.
3. 기본 묶기 모드는 준비된 BLOCK이 필요 없습니다. 라이브러리 모드라면 심볼 BLOCK 정의를 불러오거나 대표 심볼을 선택해 `BLOCK`을 만듭니다. 이름/기준점/폭을 11번 매핑과 맞추세요.
4. CAD의 Model 탭과 World UCS에서 `APPLOAD`로 `output/replace_symbols.lsp`를 로드한 후 `SYMBOL640`을 실행합니다.
5. 원본 PDF의 A/B와 같은 CAD 두 점을 찍습니다. 임의의 테두리 모서리를 페이지 모서리로 대신 찍으면 좌표가 어긋납니다.
6. 각 강조된 심볼의 선택 객체와 BLOCK을 확인합니다. `Yes`는 교체, 기본 `Skip`은 유지입니다. 기존 선이 박스 밖으로 이어지면 선택되지 않아 일부 원본이 남을 수 있습니다.
7. 결과를 열어 배선·문자·핀·축척을 점검한 뒤 `SAVEAS`로 **새 이름의 DWG**를 저장합니다.

AutoCAD Windows의 AutoLISP/Visual LISP 경로를 대상으로 스크립트를 구성했습니다. ZWCAD 호환성은 해당 버전에서 확인해야 합니다. 실제 CAD 명령 실행/새 DWG 저장은 아직 여기서 테스트하지 않았습니다.

**기본 모드는 PDFIMPORT의 실제 심볼 객체를 BLOCK으로 묶습니다.** YOLO가 BLOCK의 선을 새로 그리는 것이 아닙니다. 라이브러리 모드에서는 별도로 준비한 정확한 BLOCK이 필요합니다. 두 모드 모두 학습 라벨과 정확한 캘리브레이션이 필요하며, 심볼 밖으로 이어지는 폴리라인이나 박스 안 배선을 자동으로 완벽하게 분리하지는 못합니다.

[Autodesk PDFIMPORT 공식 문서](https://help.autodesk.com/cloudhelp/2026/ENU/AutoCAD-Core/files/GUID-02D97F82-47EC-4EE8-9429-E9CF6198EF34.htm)

출력 위치: `my_dataset/raw_images`, `my_dataset/annotations.json`, `my_dataset/exports`, `runs`, `output/detections.json`, `output/detections.reviewed.json`, `output/replace_symbols.lsp`. 학습/검출을 실행하기 전에는 이 노트북이 결과를 만들어냈다고 표시하지 않습니다.